# Review-quality model: train on Google Colab

Fine-tunes multilingual transformers to predict the review-quality score (1 to 10) and tests them on
the 400 main-run reviews. Uses the setting the team chose: "Read More" stripped from Rokomari reviews
(`RUN_TAG=noreadmore`), first 256 tokens.

**Before running:** menu **Runtime > Change runtime type > T4 GPU > Save**. Run the cells from top to
bottom. Keep this browser tab open and visible while it trains; free Colab disconnects idle sessions.

In [ ]:
# 1. Check the GPU. This should show a Tesla T4 with about 15 GB. If it fails, set the runtime type to GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the project (code + scored dataset) from GitHub, and use the team's chosen setting
!rm -rf ai-review-trends
!git clone -q -b Mastura --depth 1 https://github.com/Masturajannat/ai-review-trends.git
%cd ai-review-trends/quality_model
!pip -q install sentencepiece
%env RUN_TAG=noreadmore
!ls scripts results_noreadmore

## A. XLM-RoBERTa large (the main experiment, about 1 hour)
Three times bigger than the base model. It does not fit on an 8 GB laptop GPU, but it fits on the T4
with gradient checkpointing. Same effective batch (8 x 2 = 16) and a lower learning rate, which large
models need.

**Run only ONE of the two cells below:** A2 if Tazrian's runs showed that the product name helps
(step 1 of the plan in `README.md`), otherwise A1.


In [ ]:
# A1. without the product name
!python scripts/03_train_transformer.py --model xlm-roberta-large --lr 1e-5 --batch 8 --grad-accum 2 --grad-ckpt --warmup 0.1


In [ ]:
# A2. with the product name
!python scripts/03_train_transformer.py --model xlm-roberta-large --with-product --lr 1e-5 --batch 8 --grad-accum 2 --grad-ckpt --warmup 0.1


## B. More seeds of the large model (only if A beat the base model clearly; about 1 hour each)
Use the same command as in A, with `--seed 1` (and later `--seed 2`) and `--no-save` added. If Colab
disconnects, run the next seed in a new session; the results of finished runs are kept in the zip below.


In [ ]:
# B. example for A1; for A2 add --with-product
!python scripts/03_train_transformer.py --model xlm-roberta-large --lr 1e-5 --batch 8 --grad-accum 2 --grad-ckpt --warmup 0.1 --seed 1 --no-save


## Results

In [ ]:
# All models trained in this session, next to the baseline and the Gemini yardstick
import pandas as pd
t = pd.read_csv("results_noreadmore/model_comparison.csv")
t[["model", "MAE", "RMSE", "within 1", "within 0.5", "Spearman", "best_epoch"]].sort_values("MAE")

In [ ]:
# Seed summary of the large model (needs at least 2 seeds); use xlm-roberta-large_product after A2
!python scripts/05_seeds.py --model xlm-roberta-large


In [ ]:
# Download the results (small files). Unzip into ai-review-trends/quality_model on your laptop, then commit and push.
!zip -q -r colab_results.zip results_noreadmore -i "results_noreadmore/transformer_*" "results_noreadmore/model_comparison.csv" "results_noreadmore/seeds_*"
from google.colab import files
files.download("colab_results.zip")

## Keep the large model (only if it beat the base model)
The trained model is in `models_noreadmore/` (about 2 GB for large) and disappears when the session ends.
This copies it to your Google Drive. After A2 the folder is `xlm-roberta-large_product`.


In [ ]:
MODEL = "xlm-roberta-large"          # "xlm-roberta-large_product" after A2
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/review_quality_model"
!cp -r "models_noreadmore/{MODEL}" "/content/drive/MyDrive/review_quality_model/"
!ls -lh "/content/drive/MyDrive/review_quality_model/{MODEL}"
